# Ukraine Air Alert Time Series Analysis

## Overview
This notebook performs comprehensive time series analysis of air raid alerts across major Ukrainian cities.

### Key Metrics
- **Total alert duration per day** (minutes) - primary metric
- **Alert count** - number of separate alert events
- **Night share** - proportion of alert time during night hours (23:00-06:00)

### Cities Analyzed
- **Frontline**: Kharkiv, Sumy, Zaporizhzhia
- **Rear**: Kyiv, Odesa, Lviv
- **Strategic hub**: Dnipro

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_collection import load_raw_csv, get_city_data
from data_cleaning import (
    remove_anomalies, create_daily_series, 
    create_night_share_series, add_time_features,
    get_train_test_split, prepare_forecasting_data
)
from analysis import (
    create_analysis_summary, test_stationarity,
    analyze_weekly_pattern, analyze_monthly_pattern, analyze_yearly_trend
)
from forecasting import compare_models, train_prophet
from visualization import (
    plot_daily_duration_by_city, plot_time_bucket_analysis,
    plot_night_share, plot_heatmap, plot_forecast_comparison
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100

print('Imports loaded successfully')

## 1. Data Loading and Initial Exploration

In [ ]:
CITIES = ['Kyiv', 'Kharkiv', 'Sumy', 'Zaporizhzhia', 'Dnipro', 'Odesa', 'Lviv']

raw_df = load_raw_csv('../data/raw.csv')
print(f'Raw data shape: {raw_df.shape}')
print(f'Columns: {raw_df.columns.tolist()}')
print(f'\nFirst few rows:')
raw_df.head()

In [ ]:
cities_df = get_city_data(raw_df, cities=CITIES)
print(f'City data shape: {cities_df.shape}')
print(f'\nData by city:')
cities_df['city'].value_counts()

In [ ]:
print(f'Date range: {cities_df["started_at"].min()} to {cities_df["started_at"].max()}')
print(f'\nDuration statistics:')
cities_df['duration_min'].describe()

## 2. Data Cleaning and Anomaly Detection

In [ ]:
cleaned_df = remove_anomalies(cities_df)
print(f'Cleaned data shape: {cleaned_df.shape}')

print(f'\nRemoved {len(cities_df) - len(cleaned_df)} anomalies')
print(f'\nCleaned duration statistics:')
cleaned_df['duration_min'].describe()

## 3. Daily Time Series Construction

In [ ]:
daily_df = create_daily_series(cleaned_df)
print(f'Daily series shape: {daily_df.shape}')
print(f'\nSample data:')
daily_df.head(10)

In [ ]:
plot_daily_duration_by_city(daily_df, cities=['Kyiv', 'Kharkiv', 'Zaporizhzhia', 'Lviv'], save=False)

## 4. Time-of-Day Analysis

In [ ]:
plot_time_bucket_analysis(cleaned_df, cities=['Kyiv', 'Kharkiv', 'Zaporizhzhia', 'Lviv'], save=False)

In [ ]:
night_df = create_night_share_series(cleaned_df)
plot_night_share(night_df, cities=['Kyiv', 'Kharkiv', 'Zaporizhzhia', 'Lviv'], save=False)

## 5. Statistical Analysis

In [ ]:
for city in ['Kyiv', 'Kharkiv', 'Zaporizhzhia']:
    print(f'\n=== {city} ===')
    summary = create_analysis_summary(daily_df, city)
    print(f'  Days: {summary["n_days"]}')
    print(f'  Mean duration: {summary["mean_duration"]:.1f} min')
    print(f'  Stationary: {summary["stationarity"]["is_stationary"]}')
    print(f'  ADF p-value: {summary["stationarity"]["p_value"]:.4f}')

In [ ]:
for city in ['Kyiv', 'Kharkiv']:
    print(f'\n=== {city} Weekly Pattern ===')
    city_data = daily_df[daily_df['city'] == city]
    weekly = analyze_weekly_pattern(city_data)
    print(weekly)

In [ ]:
for city in ['Kyiv', 'Kharkiv']:
    plot_heatmap(daily_df, city, save=False)

## 6. Time Series Forecasting

In [ ]:
city = 'Kyiv'
city_data = daily_df[daily_df['city'] == city].sort_values('date')
forecast_data = prepare_forecasting_data(city_data, city)

train, test = get_train_test_split(forecast_data, test_days=30)
print(f'Train size: {len(train)}')
print(f'Test size: {len(test)}')

In [ ]:
train_series = train.set_index('date')['total_duration_min']
test_series = test.set_index('date')['total_duration_min']

comparison = compare_models(train_series, test_series, train, test)
print('\nModel Comparison:')
print(comparison[['model', 'mae', 'rmse', 'mape', 'r2']].to_string(index=False))

## 7. Multi-City Forecasting Summary

In [ ]:
results = []
for city in ['Kyiv', 'Kharkiv', 'Zaporizhzhia', 'Lviv']:
    city_data = daily_df[daily_df['city'] == city].sort_values('date')
    forecast_data = prepare_forecasting_data(city_data, city)
    
    train, test = get_train_test_split(forecast_data, test_days=30)
    train_series = train.set_index('date')['total_duration_min']
    test_series = test.set_index('date')['total_duration_min']
    
    comparison = compare_models(train_series, test_series, train, test)
    
    if not comparison.empty:
        best = comparison.loc[comparison['mape'].idxmin()]
        results.append({
            'City': city,
            'Best Model': best['model'],
            'MAPE (%)': f"{best['mape']:.1f}",
            'R²': f"{best['r2']:.3f}"
        })

results_df = pd.DataFrame(results)
print('\nMulti-City Forecast Summary:')
results_df

## 8. Key Findings

### Data Quality Issues
- Kharkiv had a 604-day anomalous record (May 2024 - Jan 2025) - removed
- Some ultra-short alerts (<1 min) represent data artifacts

### Structural Patterns
- **Frontline vs Rear divide**: Kharkiv, Sumy, Zaporizhzhia show dramatic escalation from 2025
- **Night targeting**: Rear cities (Kyiv, Odesa) have disproportionately high night share (49%, 44%)
- **Seasonal patterns**: Weekly and monthly variations exist but are secondary to geopolitical events

### Forecasting Insights
- Time series are non-stationary due to structural breaks from war escalation
- Traditional ARIMA models struggle with regime changes
- Prophet handles trend changes better but still limited by unpredictable events